In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("News Data Analysis") \
    .getOrCreate()

# Define the schema manually
schema = StructType([
    StructField("timestamp", IntegerType(), True),
    StructField("source", StringType(), True),
    StructField("archive", StringType(), True),
    StructField("id", IntegerType(), True),
    StructField("probability", FloatType(), True),
    StructField("keywords", MapType(StringType(), IntegerType()), True),
    StructField("sentiment", FloatType(), True),
    #StructField("status", StringType(), True),
    #StructField("error", StringType(), True)
])

# Load the DataFrame with the defined schema
df = spark.read.format("json").schema(schema).load("data/news/status=success")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/10 13:41:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
df01 = spark.read.format("json").schema(schema).load("data/news/status=success")
df02 = spark.read.format("json").schema(schema).load("data/news/status=error")
df03 = spark.read.format("json").schema(schema).load("data/news/status=duplicate")
df04 = spark.read.format("json").schema(schema).load("data/news/status=notnews")

total_lengt = df01.count() + df02.count() + df03.count() + df04.count()
print("Total news processed: ", total_lengt)

Total length:  1242300


In [2]:
length = df.count()
files_loaded = df.inputFiles()
print(f"Number of rows in the DataFrame: {length}")
print(f"Files loaded by Spark: {len(files_loaded)}")

df.printSchema()

Number of rows in the DataFrame: 253066
Files loaded by Spark: 2427
root
 |-- timestamp: integer (nullable = true)
 |-- source: string (nullable = true)
 |-- archive: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- probability: float (nullable = true)
 |-- keywords: map (nullable = true)
 |    |-- key: string
 |    |-- value: integer (valueContainsNull = true)
 |-- sentiment: float (nullable = true)



In [3]:
df.show()

+---------+----------------+--------------------+------+-----------+--------------------+-----------+
|timestamp|          source|             archive|    id|probability|            keywords|  sentiment|
+---------+----------------+--------------------+------+-----------+--------------------+-----------+
|   201709|Correio da Manhã|https://arquivo.p...|701644| 0.58617663|{Amazon Carlos Ro...|  0.6422417|
|   201709|Correio da Manhã|https://arquivo.p...|700910|  0.5093835|{Finge -> 1, aero...| -0.5444501|
|   201709|Correio da Manhã|https://arquivo.p...|700927|    0.57878|{falha -> 2, Fing...|-0.56717324|
|   201709|Correio da Manhã|https://arquivo.p...|700218|  0.5356582|{Impresso -> 1, P...|-0.54087096|
|   201709|Correio da Manhã|https://arquivo.p...|700475|  0.5083431|{restrição -> 2, ...|-0.33665743|
|   201709|            SAPO|https://arquivo.p...|701172| 0.55286324|{Astúrias -> 1, A...|-0.36502635|
|   201709|Correio da Manhã|https://arquivo.p...|700180| 0.52512944|{PS -> 1, alvo

In [4]:
from pyspark.sql import functions as F

keyword = "Galp"

# Check if the key "X" exists in the "keywords" map
#df_with_key = df.filter(df["keywords"].getItem(keyword).isNotNull())
df_with_key = df.filter(F.col("keywords").getItem(keyword).isNotNull() & (F.col("keywords").getItem(keyword) > 4))

length = df_with_key.count()
print(f"Number of rows in the DataFrame: {length}")

df_with_key.show(truncate=False)

Number of rows in the DataFrame: 627


+---------+------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+-----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [5]:
# Process data
result = (
    df_with_key.rdd
    .flatMap(lambda row: [
        (key, (value,
               {row["timestamp"]: value},
               row["sentiment"]*value,
               {row["source"]: 1},
               [row["archive"]])) for key, value in row["keywords"].items()
    ])
    .reduceByKey(lambda a, b: (
        a[0] + b[0],  # Sum count values
        {ts: a[1].get(ts, 0) + b[1].get(ts, 0) for ts in set(a[1]) | set(b[1])},  # Merge timestamp counts
        a[2] + b[2],  # Sum sentiment values
        {source: a[3].get(source, 0) + b[3].get(source, 0) for source in set(a[3]) | set(b[3])},  # Merge source counts
        a[4] + b[4]  # Merge archive lists
    ))
    .collect()
)

# Convert to dictionary
output = {key: {"count": value[0],
                "date": value[1],
                "sentiment": value[2]/value[0],
                "source": value[3],
                "news": value[4]} for key, value in result}

# Print output
#output

sorted_dict = dict(sorted(output.items(), key=lambda item: item[1]["count"], reverse=True))
#sorted_dict


OpenJDK 64-Bit Server VM warning: CodeCache is full. Compiler has been disabled.
OpenJDK 64-Bit Server VM warning: Try increasing the code cache size using -XX:ReservedCodeCacheSize=


CodeCache: size=131072Kb used=31646Kb max_used=31646Kb free=99425Kb
 bounds [0x000000010698c000, 0x00000001088ac000, 0x000000010e98c000]
 total_blobs=12270 nmethods=11312 adapters=870
 compilation: disabled (not enough contiguous free space left)


In [9]:
print(len(output))

19714


In [6]:
import json

In [7]:
file_path = 'data/keywords_test.json'

# Save the dictionary to a JSON file
with open(file_path, 'w') as json_file:
    json.dump(output, json_file, indent=4)

print(f"JSON data saved to {file_path}")

# Load the JSON data back into a Python dictionary
with open(file_path, 'r') as json_file:
    loaded_data = json.load(json_file)

# Print loaded data
print("Loaded data:")
#loaded_data

JSON data saved to data/keywords_test.json
Loaded data:


In [8]:
#loaded_data["Comerciantes"]["news"][2]